In [53]:
import pandas as pd
import psycopg

In [54]:
predictions = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")

predictions["Date"] = pd.to_datetime(predictions["Date"])

print("Prediction rows:", len(predictions))
print("Date range:", predictions["Date"].min(), "to", predictions["Date"].max())

predictions.head()

Prediction rows: 508
Date range: 2024-09-02 00:00:00 to 2026-08-21 00:00:00


,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2024-09-02,Monday,Long,61%,Moderate Up,Trend Continuation,Invalid,NaN,Positive ORG followed an overnight sell-side s...
1,2024-09-03,Tuesday,Short,63%,Strong Down,Trend Continuation,Short,True,Large negative ORG accompanied a persistent de...
2,2024-09-04,Wednesday,Short,66%,Strong Down,Trend Continuation,Long,False,Large negative ORG followed an exceptionally w...
3,2024-09-05,Thursday,Short,59%,Moderate Down,Balance,Long,False,Negative ORG developed within a choppy overnig...
4,2024-09-06,Friday,Long,62%,Moderate Up,Exhaustion,Short,False,Nearly flat ORG followed a broad overnight ran...


In [55]:
conn = psycopg.connect("dbname=dailyedge_development")

query = """
SELECT timestamp, open, high, low, close, volume
FROM CANDLES
WHERE timestamp >= '2025-09-01 08:30:00'
  AND timestamp <= '2026-07-07 15:15:00'
  AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
ORDER BY timestamp
"""

candles = pd.read_sql(query, conn)
conn.close()

candles["timestamp"] = pd.to_datetime(candles["timestamp"])
candles["Date"] = candles["timestamp"].dt.normalize()

print("RTH candles:", len(candles))
print("Sessions:", candles["Date"].nunique())
print("Date range:", candles["Date"].min(), "to", candles["Date"].max())

/tmp/ipykernel_16638/1973621230.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql(query, conn)


RTH candles: 87179
Sessions: 219
Date range: 2025-09-01 00:00:00 to 2026-07-07 00:00:00


In [56]:
# Match prediction dates to available candle sessions

prediction_dates = set(predictions["Date"].dropna())
candle_dates = set(candles["Date"].dropna())

matched_dates = prediction_dates & candle_dates

matched_predictions = predictions[
    predictions["Date"].isin(matched_dates)
].copy()

matched_candles = candles[
    candles["Date"].isin(matched_dates)
].copy()

session_validation = (
    matched_candles.groupby("Date")
    .agg(
        first_time=("timestamp", "min"),
        last_time=("timestamp", "max"),
        candle_count=("timestamp", "size")
    )
)

session_validation["has_open"] = (
    session_validation["first_time"].dt.time ==
    pd.Timestamp("08:30:00").time()
)

session_validation["has_close"] = (
    session_validation["last_time"].dt.time ==
    pd.Timestamp("15:15:00").time()
)

print("Prediction dates:", len(prediction_dates))
print("Candle dates:", len(candle_dates))
print("Matched dates:", len(matched_dates))

print("Matched prediction/session dates:", len(matched_predictions))
print("Sessions with 08:30 open:", session_validation["has_open"].sum())
print("Sessions with 15:15 close:", session_validation["has_close"].sum())

print(
    "Fully valid sessions:",
    (session_validation["has_open"] & session_validation["has_close"]).sum()
)

session_validation[
    ~(session_validation["has_open"] & session_validation["has_close"])
]

Prediction dates: 507
Candle dates: 219
Matched dates: 218
Matched prediction/session dates: 218
Sessions with 08:30 open: 218
Sessions with 15:15 close: 209
Fully valid sessions: 209


,first_time,last_time,candle_count,has_open,has_close
Date,,,,,
2025-09-01,2025-09-01 08:30:00,2025-09-01 11:59:00,210,True,False
2025-11-27,2025-11-27 08:30:00,2025-11-27 11:59:00,210,True,False
2025-11-28,2025-11-28 08:30:00,2025-11-28 12:14:00,225,True,False
2025-12-24,2025-12-24 08:30:00,2025-12-24 12:14:00,225,True,False
2026-01-19,2026-01-19 08:30:00,2026-01-19 11:59:00,210,True,False
2026-02-16,2026-02-16 08:30:00,2026-02-16 11:59:00,210,True,False
2026-05-25,2026-05-25 08:30:00,2026-05-25 11:59:00,210,True,False
2026-06-19,2026-06-19 08:30:00,2026-06-19 11:59:00,210,True,False
2026-07-03,2026-07-03 08:30:00,2026-07-03 11:59:00,210,True,False


In [57]:
valid_dates = session_validation[
    session_validation["has_open"] &
    session_validation["has_close"]
].index

study_predictions = matched_predictions[
    matched_predictions["Date"].isin(valid_dates)
].copy()

study_candles = matched_candles[
    matched_candles["Date"].isin(valid_dates)
].copy()

print("Study prediction rows:", len(study_predictions))
print("Study candle sessions:", study_candles["Date"].nunique())
print("Study RTH candles:", len(study_candles))

print("\nPrediction results:")
print(study_predictions["Result"].value_counts(dropna=False))

Study prediction rows: 209
Study candle sessions: 209
Study RTH candles: 84853

Prediction results:
Result
Short      107
Long       101
Neither      1
Name: count, dtype: int64


In [58]:
def evaluate_continuous_trail_long(day_df, trail_distance, target_distance):
    open_price = day_df.iloc[0]["open"]

    stop_price = open_price - trail_distance
    target_price = open_price + target_distance

    for _, candle in day_df.iterrows():
        active_stop = stop_price

        # Gap/open through the active stop
        if candle["open"] <= active_stop:
            return "Stopped", active_stop - open_price

        stop_hit = candle["low"] <= active_stop
        target_hit = candle["high"] >= target_price

        # 1-minute OHLC cannot tell us which occurred first
        if stop_hit and target_hit:
            return "Unknown", None

        if target_hit:
            return "Target", target_distance

        if stop_hit:
            return "Stopped", active_stop - open_price

        # Trail updates only after this candle survives
        stop_price = max(
            active_stop,
            candle["high"] - trail_distance
        )

    # Still open at the end of RTH
    final_close = day_df.iloc[-1]["close"]
    return "Close", final_close - open_price


def evaluate_continuous_trail_short(day_df, trail_distance, target_distance):
    open_price = day_df.iloc[0]["open"]

    stop_price = open_price + trail_distance
    target_price = open_price - target_distance

    for _, candle in day_df.iterrows():
        active_stop = stop_price

        # Gap/open through the active stop
        if candle["open"] >= active_stop:
            return "Stopped", open_price - active_stop

        stop_hit = candle["high"] >= active_stop
        target_hit = candle["low"] <= target_price

        # 1-minute OHLC cannot tell us which occurred first
        if stop_hit and target_hit:
            return "Unknown", None

        if target_hit:
            return "Target", target_distance

        if stop_hit:
            return "Stopped", open_price - active_stop

        # Trail updates only after this candle survives
        stop_price = min(
            active_stop,
            candle["low"] + trail_distance
        )

    # Still open at the end of RTH
    final_close = day_df.iloc[-1]["close"]
    return "Close", open_price - final_close

In [59]:
def evaluate_one_time_be_long(day_df, stop_distance, target_distance):
    open_price = day_df.iloc[0]["open"]

    stop_price = open_price - stop_distance
    target_price = open_price + target_distance
    be_trigger = open_price + stop_distance

    moved_to_be = False

    for _, candle in day_df.iterrows():
        active_stop = stop_price

        # Gap/open through the active stop
        if candle["open"] <= active_stop:
            return "Breakeven" if moved_to_be else "Stopped", active_stop - open_price

        stop_hit = candle["low"] <= active_stop
        target_hit = candle["high"] >= target_price
        be_hit = (not moved_to_be) and (candle["high"] >= be_trigger)

        # Existing stop and target both touched in same candle
        if stop_hit and target_hit:
            return "Unknown", None

        if target_hit:
            return "Target", target_distance

        if stop_hit:
            return "Breakeven" if moved_to_be else "Stopped", active_stop - open_price

        # If BE trigger is reached, move stop once for future candles
        if be_hit:
            stop_price = open_price
            moved_to_be = True

    final_close = day_df.iloc[-1]["close"]
    return "Close", final_close - open_price


def evaluate_one_time_be_short(day_df, stop_distance, target_distance):
    open_price = day_df.iloc[0]["open"]

    stop_price = open_price + stop_distance
    target_price = open_price - target_distance
    be_trigger = open_price - stop_distance

    moved_to_be = False

    for _, candle in day_df.iterrows():
        active_stop = stop_price

        # Gap/open through the active stop
        if candle["open"] >= active_stop:
            return "Breakeven" if moved_to_be else "Stopped", open_price - active_stop

        stop_hit = candle["high"] >= active_stop
        target_hit = candle["low"] <= target_price
        be_hit = (not moved_to_be) and (candle["low"] <= be_trigger)

        # Existing stop and target both touched in same candle
        if stop_hit and target_hit:
            return "Unknown", None

        if target_hit:
            return "Target", target_distance

        if stop_hit:
            return "Breakeven" if moved_to_be else "Stopped", open_price - active_stop

        # If BE trigger is reached, move stop once for future candles
        if be_hit:
            stop_price = open_price
            moved_to_be = True

    final_close = day_df.iloc[-1]["close"]
    return "Close", open_price - final_close

In [60]:
stop_target_pairs = [
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 80),
    (50, 100),
    (65, 100),
    (75, 100),
    (100, 100),
    (100, 50),
    (100, 25),
    (75, 150),
    (100, 150)
]

stop_target_pairs

[(15, 25),
 (25, 50),
 (35, 70),
 (50, 70),
 (50, 80),
 (50, 100),
 (65, 100),
 (75, 100),
 (100, 100),
 (100, 50),
 (100, 25),
 (75, 150),
 (100, 150)]

In [61]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

In [62]:
continuous_results = []

for stop_distance, target_distance in stop_target_pairs:
    for _, prediction in study_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Bias"]

        day_df = study_candles[
            study_candles["Date"] == date
        ]

        if direction == "Long":
            status, pnl = evaluate_continuous_trail_long(
                day_df,
                trail_distance=stop_distance,
                target_distance=target_distance
            )

        elif direction == "Short":
            status, pnl = evaluate_continuous_trail_short(
                day_df,
                trail_distance=stop_distance,
                target_distance=target_distance
            )

        else:
            status, pnl = "Unknown", None

        continuous_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Stop": stop_distance,
            "Target": target_distance,
            "Pair": f"{stop_distance}/{target_distance}",
            "Status": status,
            "PnL": pnl
        })

continuous_trades = pd.DataFrame(continuous_results)

print("Total simulated trades:", len(continuous_trades))

print("\nTrades per pair:")
print(continuous_trades["Pair"].value_counts().reindex(
    [f"{s}/{t}" for s, t in stop_target_pairs]
))

print("\nUnknowns per pair:")
print(
    continuous_trades[
        continuous_trades["Status"] == "Unknown"
    ]["Pair"].value_counts().reindex(
        [f"{s}/{t}" for s, t in stop_target_pairs],
        fill_value=0
    )
)

Total simulated trades: 2717

Trades per pair:
Pair
15/25      209
25/50      209
35/70      209
50/70      209
50/80      209
50/100     209
65/100     209
75/100     209
100/100    209
100/50     209
100/25     209
75/150     209
100/150    209
Name: count, dtype: int64

Unknowns per pair:
Pair
15/25      41
25/50       4
35/70       1
50/70       1
50/80       1
50/100      2
65/100      2
75/100      2
100/100     1
100/50      0
100/25      0
75/150      0
100/150     0
Name: count, dtype: int64


In [63]:
one_time_be_results = []

for stop_distance, target_distance in stop_target_pairs:
    for _, prediction in study_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Bias"]

        day_df = study_candles[
            study_candles["Date"] == date
        ]

        if direction == "Long":
            status, pnl = evaluate_one_time_be_long(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        elif direction == "Short":
            status, pnl = evaluate_one_time_be_short(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        else:
            status, pnl = "Unknown", None

        one_time_be_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Stop": stop_distance,
            "Target": target_distance,
            "Pair": f"{stop_distance}/{target_distance}",
            "Status": status,
            "PnL": pnl
        })

one_time_be_trades = pd.DataFrame(one_time_be_results)

print("Total simulated trades:", len(one_time_be_trades))

print("\nTrades per pair:")
print(
    one_time_be_trades["Pair"]
    .value_counts()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

print("\nUnknowns per pair:")
print(
    one_time_be_trades[
        one_time_be_trades["Status"] == "Unknown"
    ]["Pair"]
    .value_counts()
    .reindex(
        [f"{s}/{t}" for s, t in stop_target_pairs],
        fill_value=0
    )
)

Total simulated trades: 2717

Trades per pair:
Pair
15/25      209
25/50      209
35/70      209
50/70      209
50/80      209
50/100     209
65/100     209
75/100     209
100/100    209
100/50     209
100/25     209
75/150     209
100/150    209
Name: count, dtype: int64

Unknowns per pair:
Pair
15/25      39
25/50       1
35/70       0
50/70       1
50/80       1
50/100      1
65/100      1
75/100      1
100/100     0
100/50      0
100/25      0
75/150      0
100/150     0
Name: count, dtype: int64


In [64]:
continuous_expectancy_matrix = (
    continuous_trades
    .dropna(subset=["PnL"])
    .groupby(["Pair", "Day"])["PnL"]
    .mean()
    .unstack("Day")
    .reindex(
        index=[f"{s}/{t}" for s, t in stop_target_pairs],
        columns=weekday_order
    )
)

resolved_continuous = continuous_trades.dropna(subset=["PnL"])

valid_sessions = (
    resolved_continuous
    .groupby("Pair")
    .size()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

average_loss = (
    resolved_continuous[
        resolved_continuous["PnL"] < 0
    ]
    .groupby("Pair")["PnL"]
    .mean()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

continuous_expectancy_matrix.insert(
    0,
    "Valid Sessions",
    valid_sessions
)

continuous_expectancy_matrix.insert(
    1,
    "Average Loss",
    average_loss
)

continuous_expectancy_matrix.round(2)

Day,Valid Sessions,Average Loss,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,,,
15/25,168,-14.50,5.78,0.30,4.86,1.08,4.94
25/50,205,-20.54,6.73,3.24,14.71,0.88,7.05
35/70,208,-23.93,11.05,1.57,14.06,-1.21,7.94
50/70,208,-30.61,10.50,4.82,9.91,-3.14,10.26
50/80,208,-30.64,11.26,5.11,9.45,-0.70,6.69
50/100,207,-30.64,10.34,2.43,5.89,-3.26,9.65
65/100,207,-38.17,14.57,4.49,4.92,-4.90,8.38
75/100,207,-45.90,12.95,3.89,-0.49,-9.11,9.53
100/100,208,-56.86,9.99,19.96,14.75,-10.88,19.17


In [65]:
one_time_be_expectancy_matrix = (
    one_time_be_trades
    .dropna(subset=["PnL"])
    .groupby(["Pair", "Day"])["PnL"]
    .mean()
    .unstack("Day")
    .reindex(
        index=[f"{s}/{t}" for s, t in stop_target_pairs],
        columns=weekday_order
    )
)

resolved_one_time_be = one_time_be_trades.dropna(subset=["PnL"])

valid_sessions = (
    resolved_one_time_be
    .groupby("Pair")
    .size()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

average_loss = (
    resolved_one_time_be[
        resolved_one_time_be["PnL"] < 0
    ]
    .groupby("Pair")["PnL"]
    .mean()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

one_time_be_expectancy_matrix.insert(
    0,
    "Valid Sessions",
    valid_sessions
)

one_time_be_expectancy_matrix.insert(
    1,
    "Average Loss",
    average_loss
)

one_time_be_expectancy_matrix.round(2)

Day,Valid Sessions,Average Loss,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,,,
15/25,170,-15.00,6.83,-0.26,5.00,0.44,4.48
25/50,208,-25.00,5.62,3.89,15.12,0.00,5.13
35/70,209,-35.00,8.54,7.78,15.47,-1.71,1.79
50/70,208,-50.00,7.56,13.11,13.95,-2.93,13.42
50/80,208,-50.00,10.00,12.95,14.88,0.49,10.26
50/100,208,-50.00,10.98,10.23,9.30,0.00,10.26
65/100,208,-65.00,7.32,9.09,7.67,-8.78,26.15
75/100,208,-74.51,0.61,10.80,16.28,-4.07,29.32
100/100,209,-97.79,-7.32,25.49,16.28,-4.07,42.37


In [66]:
def evaluate_continuous_trail_with_time(
    day_df,
    direction,
    trail_distance=50,
    target_distance=100
):
    open_price = day_df.iloc[0]["open"]

    if direction == "Long":
        stop_price = open_price - trail_distance
        target_price = open_price + target_distance

        for _, candle in day_df.iterrows():
            active_stop = stop_price

            if candle["open"] <= active_stop:
                return "Stopped", active_stop - open_price, candle["timestamp"]

            stop_hit = candle["low"] <= active_stop
            target_hit = candle["high"] >= target_price

            if stop_hit and target_hit:
                return "Unknown", None, candle["timestamp"]

            if target_hit:
                return "Target", target_distance, candle["timestamp"]

            if stop_hit:
                return "Stopped", active_stop - open_price, candle["timestamp"]

            stop_price = max(
                active_stop,
                candle["high"] - trail_distance
            )

        final_close = day_df.iloc[-1]["close"]
        return "Close", final_close - open_price, day_df.iloc[-1]["timestamp"]

    elif direction == "Short":
        stop_price = open_price + trail_distance
        target_price = open_price - target_distance

        for _, candle in day_df.iterrows():
            active_stop = stop_price

            if candle["open"] >= active_stop:
                return "Stopped", open_price - active_stop, candle["timestamp"]

            stop_hit = candle["high"] >= active_stop
            target_hit = candle["low"] <= target_price

            if stop_hit and target_hit:
                return "Unknown", None, candle["timestamp"]

            if target_hit:
                return "Target", target_distance, candle["timestamp"]

            if stop_hit:
                return "Stopped", open_price - active_stop, candle["timestamp"]

            stop_price = min(
                active_stop,
                candle["low"] + trail_distance
            )

        final_close = day_df.iloc[-1]["close"]
        return "Close", open_price - final_close, day_df.iloc[-1]["timestamp"]

    return "Unknown", None, None

In [67]:
timed_results = []

for _, prediction in study_predictions.iterrows():
    date = prediction["Date"]
    direction = prediction["Bias"]

    day_df = study_candles[
        study_candles["Date"] == date
    ]

    status, pnl, exit_time = evaluate_continuous_trail_with_time(
        day_df,
        direction,
        trail_distance=50,
        target_distance=100
    )

    timed_results.append({
        "Date": date,
        "Day": prediction["Day"],
        "Bias": direction,
        "Status": status,
        "PnL": pnl,
        "Exit_Time": exit_time
    })

timed_trades = pd.DataFrame(timed_results)

losing_trades = timed_trades[
    timed_trades["PnL"] < 0
].copy()

losing_trades["Minutes_From_Open"] = (
    (
        losing_trades["Exit_Time"]
        - losing_trades["Exit_Time"].dt.normalize()
        - pd.Timedelta(hours=8, minutes=30)
    )
    .dt.total_seconds()
    / 60
)

bins = [0, 5, 10, 15, 20, 25, 30]

labels = [
    "0–4 min",
    "5–9 min",
    "10–14 min",
    "15–19 min",
    "20–24 min",
    "25–29 min"
]

losing_trades["Exit_Range"] = pd.cut(
    losing_trades["Minutes_From_Open"],
    bins=bins,
    labels=labels,
    right=False
)

loss_time_matrix = (
    losing_trades
    .groupby(["Day", "Exit_Range"], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=weekday_order,
        columns=labels,
        fill_value=0
    )
)

loss_time_matrix.insert(
    0,
    "Total Losses",
    loss_time_matrix.sum(axis=1)
)

loss_time_matrix

Exit_Range,Total Losses,0–4 min,5–9 min,10–14 min,15–19 min,20–24 min,25–29 min
Day,,,,,,,
Monday,22,19,3,0,0,0,0
Tuesday,25,19,3,3,0,0,0
Wednesday,22,17,3,1,0,0,1
Thursday,26,21,2,2,1,0,0
Friday,21,18,1,0,0,2,0


In [68]:
def evaluate_one_time_be_add_long(
    day_df,
    stop_distance,
    target_distance
):
    open_price = day_df.iloc[0]["open"]

    initial_stop = open_price - stop_distance
    add_price = open_price + stop_distance
    target_price = open_price + target_distance

    added = False

    for _, candle in day_df.iterrows():

        # Before the add triggers
        if not added:
            stop_hit = candle["low"] <= initial_stop
            add_hit = candle["high"] >= add_price
            target_hit = candle["high"] >= target_price

            # Intrabar ordering cannot be determined
            if stop_hit and (add_hit or target_hit):
                return "Unknown", None

            if target_hit:
                return "Target", target_distance

            if add_hit:
                added = True
                continue

            if stop_hit:
                return "Stopped", -stop_distance

        # After the add:
        # Unit 1 stop = original entry
        # Unit 2 stop = original entry
        # Both target the original target
        else:
            combined_stop = open_price

            stop_hit = candle["low"] <= combined_stop
            target_hit = candle["high"] >= target_price

            # Cannot determine whether target or stop happened first
            if stop_hit and target_hit:
                return "Unknown", None

            if target_hit:
                unit_1_pnl = target_distance
                unit_2_pnl = target_distance - stop_distance

                return "Target", unit_1_pnl + unit_2_pnl

            if stop_hit:
                unit_1_pnl = 0
                unit_2_pnl = -stop_distance

                return "Stopped", unit_1_pnl + unit_2_pnl

    # Session ends before either stop or target
    final_close = day_df.iloc[-1]["close"]

    if not added:
        pnl = final_close - open_price
    else:
        unit_1_pnl = final_close - open_price
        unit_2_pnl = final_close - add_price
        pnl = unit_1_pnl + unit_2_pnl

    return "Close", pnl

In [69]:
def evaluate_one_time_be_add_short(
    day_df,
    stop_distance,
    target_distance
):
    open_price = day_df.iloc[0]["open"]

    initial_stop = open_price + stop_distance
    add_price = open_price - stop_distance
    target_price = open_price - target_distance

    added = False

    for _, candle in day_df.iterrows():

        # Before the add triggers
        if not added:
            stop_hit = candle["high"] >= initial_stop
            add_hit = candle["low"] <= add_price
            target_hit = candle["low"] <= target_price

            # Intrabar ordering cannot be determined
            if stop_hit and (add_hit or target_hit):
                return "Unknown", None

            if target_hit:
                return "Target", target_distance

            if add_hit:
                added = True
                continue

            if stop_hit:
                return "Stopped", -stop_distance

        # After the add:
        # Unit 1 stop = original entry
        # Unit 2 stop = original entry
        # Both target the original target
        else:
            combined_stop = open_price

            stop_hit = candle["high"] >= combined_stop
            target_hit = candle["low"] <= target_price

            # Cannot determine whether target or stop happened first
            if stop_hit and target_hit:
                return "Unknown", None

            if target_hit:
                unit_1_pnl = target_distance
                unit_2_pnl = target_distance - stop_distance

                return "Target", unit_1_pnl + unit_2_pnl

            if stop_hit:
                unit_1_pnl = 0
                unit_2_pnl = -stop_distance

                return "Stopped", unit_1_pnl + unit_2_pnl

    # Session ends before either stop or target
    final_close = day_df.iloc[-1]["close"]

    if not added:
        pnl = open_price - final_close
    else:
        unit_1_pnl = open_price - final_close
        unit_2_pnl = add_price - final_close
        pnl = unit_1_pnl + unit_2_pnl

    return "Close", pnl

In [70]:
one_time_be_add_results = []

for stop_distance, target_distance in stop_target_pairs:
    for _, prediction in study_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Bias"]

        day_df = study_candles[
            study_candles["Date"] == date
        ]

        if direction == "Long":
            status, pnl = evaluate_one_time_be_add_long(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        elif direction == "Short":
            status, pnl = evaluate_one_time_be_add_short(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        else:
            status, pnl = "Unknown", None

        one_time_be_add_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Stop": stop_distance,
            "Target": target_distance,
            "Pair": f"{stop_distance}/{target_distance}",
            "Status": status,
            "PnL": pnl
        })

one_time_be_add_trades = pd.DataFrame(one_time_be_add_results)

print("Total simulated trades:", len(one_time_be_add_trades))

print("\nTrades per pair:")
print(
    one_time_be_add_trades["Pair"]
    .value_counts()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

print("\nUnknowns per pair:")
print(
    one_time_be_add_trades[
        one_time_be_add_trades["Status"] == "Unknown"
    ]["Pair"]
    .value_counts()
    .reindex(
        [f"{s}/{t}" for s, t in stop_target_pairs],
        fill_value=0
    )
)

Total simulated trades: 2717

Trades per pair:
Pair
15/25      209
25/50      209
35/70      209
50/70      209
50/80      209
50/100     209
65/100     209
75/100     209
100/100    209
100/50     209
100/25     209
75/150     209
100/150    209
Name: count, dtype: int64

Unknowns per pair:
Pair
15/25      64
25/50      17
35/70       6
50/70       3
50/80       4
50/100      3
65/100      1
75/100      1
100/100     0
100/50      0
100/25      0
75/150      0
100/150     0
Name: count, dtype: int64


In [71]:
one_time_be_add_expectancy_matrix = (
    one_time_be_add_trades
    .dropna(subset=["PnL"])
    .groupby(["Pair", "Day"])["PnL"]
    .mean()
    .unstack("Day")
    .reindex(
        index=[f"{s}/{t}" for s, t in stop_target_pairs],
        columns=weekday_order
    )
)

resolved_one_time_be_add = (
    one_time_be_add_trades
    .dropna(subset=["PnL"])
)

valid_sessions = (
    resolved_one_time_be_add
    .groupby("Pair")
    .size()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

average_loss = (
    resolved_one_time_be_add[
        resolved_one_time_be_add["PnL"] < 0
    ]
    .groupby("Pair")["PnL"]
    .mean()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

one_time_be_add_expectancy_matrix.insert(
    0,
    "Valid Sessions",
    valid_sessions
)

one_time_be_add_expectancy_matrix.insert(
    1,
    "Average Loss",
    average_loss
)

one_time_be_add_expectancy_matrix.round(2)

Day,Valid Sessions,Average Loss,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,,,
15/25,145,-15.00,10.19,1.97,8.94,2.78,6.20
25/50,192,-25.00,10.71,2.33,17.44,5.41,11.03
35/70,203,-35.00,14.36,12.44,19.17,0.00,2.69
50/70,206,-50.00,8.05,17.73,14.76,-4.15,13.16
50/80,205,-50.00,12.68,19.30,16.43,2.44,12.37
50/100,206,-50.00,18.29,15.12,4.65,3.66,13.16
65/100,208,-65.00,13.05,10.68,3.14,-6.46,27.31
75/100,208,-74.64,2.44,10.23,11.63,-10.17,28.04
100/100,209,-97.79,-7.32,25.49,16.28,-4.07,42.37


In [74]:
reverse_thursday_results = []

for stop_distance, target_distance in stop_target_pairs:
    for _, prediction in study_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Effective_Bias"]

        day_df = study_candles[
            study_candles["Date"] == date
        ]

        if direction == "Long":
            status, pnl = evaluate_one_time_be_add_long(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        elif direction == "Short":
            status, pnl = evaluate_one_time_be_add_short(
                day_df,
                stop_distance=stop_distance,
                target_distance=target_distance
            )

        else:
            status, pnl = "Unknown", None

        reverse_thursday_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Stop": stop_distance,
            "Target": target_distance,
            "Pair": f"{stop_distance}/{target_distance}",
            "Status": status,
            "PnL": pnl
        })

reverse_thursday_trades = pd.DataFrame(reverse_thursday_results)

reverse_thursday_expectancy_matrix = (
    reverse_thursday_trades
    .dropna(subset=["PnL"])
    .groupby(["Pair", "Day"])["PnL"]
    .mean()
    .unstack("Day")
    .reindex(
        index=[f"{s}/{t}" for s, t in stop_target_pairs],
        columns=weekday_order
    )
)

resolved_reverse_thursday = (
    reverse_thursday_trades
    .dropna(subset=["PnL"])
)

valid_sessions = (
    resolved_reverse_thursday
    .groupby("Pair")
    .size()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

average_loss = (
    resolved_reverse_thursday[
        resolved_reverse_thursday["PnL"] < 0
    ]
    .groupby("Pair")["PnL"]
    .mean()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

reverse_thursday_expectancy_matrix.insert(0, "Valid Sessions", valid_sessions)
reverse_thursday_expectancy_matrix.insert(1, "Average Loss", average_loss)

reverse_thursday_expectancy_matrix.round(2)

Day,Valid Sessions,Average Loss,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,,,
15/25,145,-15.00,10.19,1.97,8.94,-0.19,6.20
25/50,192,-25.00,10.71,2.33,17.44,9.46,11.03
35/70,204,-35.00,14.36,12.44,19.17,14.36,2.69
50/70,206,-50.00,8.05,17.73,14.76,3.66,13.16
50/80,205,-50.00,12.68,19.30,16.43,0.73,12.37
50/100,206,-50.00,18.29,15.12,4.65,8.54,13.16
65/100,208,-65.00,13.05,10.68,3.14,12.20,27.31
75/100,208,-74.27,2.44,10.23,11.63,7.73,28.04
100/100,209,-97.01,-7.32,25.49,16.28,4.07,42.37


In [72]:
be_add_success_matrix = (
    one_time_be_add_trades
    .dropna(subset=["PnL"])
    .assign(Success=lambda df: df["PnL"] > 0)
    .groupby(["Pair", "Day"])["Success"]
    .mean()
    .mul(100)
    .unstack("Day")
    .reindex(
        index=[f"{s}/{t}" for s, t in stop_target_pairs],
        columns=weekday_order
    )
)

valid_sessions = (
    one_time_be_add_trades
    .dropna(subset=["PnL"])
    .groupby("Pair")
    .size()
    .reindex([f"{s}/{t}" for s, t in stop_target_pairs])
)

be_add_success_matrix.insert(
    0,
    "Valid Sessions",
    valid_sessions
)

be_add_success_matrix.round(2)

Day,Valid Sessions,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,,
15/25,145,59.26,42.42,57.58,44.44,52.00
25/50,192,40.00,32.56,48.84,35.14,41.18
35/70,203,38.46,35.56,40.48,26.32,28.21
50/70,206,43.90,50.00,47.62,34.15,47.37
50/80,205,41.46,44.19,42.86,34.15,39.47
50/100,206,34.15,32.56,27.91,26.83,31.58
65/100,208,39.02,38.64,34.88,29.27,46.15
75/100,208,39.02,43.18,44.19,34.15,51.28
100/100,209,46.34,62.22,58.14,48.78,69.23
